In [4]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

import pyro
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoNormal
from pyro.optim import Adam, Adagrad
import pyro.distributions as dist
import pyro.optim as optim
import json

In [5]:
# helpers
def binomial(n_samples, theta_like=0.7, seed=20):
    np.random.seed(seed)
    data = np.random.binomial(n_samples, theta_like)
    return data

def load_config(config_file="config.json"):

    with open(config_file, 'r') as f:
        config = json.load(f)
    return config

def _logistic_normal_mean_std_mc(mu, sigma, n_samples=10_000, seed=0):
    mu = float(np.squeeze(mu))
    sigma = float(np.squeeze(sigma))
    rng = np.random.default_rng(seed)
    z = rng.normal(loc=mu, scale=sigma, size=n_samples)
    theta = expit(z)
    return float(theta.mean()), float(theta.std(ddof=1))

def _trace_locscale_to_theta_moments(loc_trace, scale_trace, n_samples=10_000, seed=0):
    """Batched transformation using JAX for maximum speed."""
    # originally looped element-by-element then we tried vector-broadcasting
    loc_trace = np.asarray(loc_trace, dtype=np.float32)
    scale_trace = np.asarray(scale_trace, dtype=np.float32)
    
    T = loc_trace.shape[0]
    batch_size = 5000  # Process 5000 timepoints at once (50M elements = ~200MB)
    
    out_mean = []
    out_std = []
    
    key = jax.random.PRNGKey(seed)
    
    for start_idx in range(0, T, batch_size):
        end_idx = min(start_idx + batch_size, T)
        batch_loc = loc_trace[start_idx:end_idx]
        batch_scale = scale_trace[start_idx:end_idx]
        
        # Use JAX's random generation - much faster
        key, subkey = jax.random.split(key)
        z = jax.random.normal(subkey, shape=(batch_loc.shape[0], n_samples)) * batch_scale[:, None] + batch_loc[:, None]
        theta = jax.nn.sigmoid(z)  # JAX's optimized sigmoid
        
        out_mean.append(np.mean(theta, axis=1))
        out_std.append(np.std(theta, axis=1, ddof=1))
    
    return np.concatenate(out_mean), np.concatenate(out_std)

data_gen = binomial


In [6]:
config_file = "../beta_config.json"
config = load_config(config_file)

theta_like = config['theta_like']
alpha_prior = config['alpha_prior']
beta_prior = config['beta_prior']
n_samples = config['n_samples']
max_iters = 100_000
adam_step = 5e-4 # the "default"

data = data_gen(n_samples) if n_samples > 0 else 0
# print parameters
print("prior: Beta({}, {})".format(alpha_prior, beta_prior))
print("Likelihood: Binomial({})".format(theta_like))
print(data)
# true_post = beta_posterior(data, n_samples, alpha_prior, beta_prior)
# print("True posterior: Beta({}, {:.2f})".format(true_post['alpha_post'], true_post['beta_post']) + " with mean {:.2f} and std {:.2f}".format(true_post['mean_post'], true_post['std_post']))
# Note that our plots do compare to the "best variational approximation"

prior: Beta(0.5, 0.6)
Likelihood: Binomial(0.7)
7


In [17]:
def run_restart(seed):
    np.random.seed(seed)
    pyro.set_rng_seed(seed)

    def single_model(y):
        mu = pyro.sample("theta", dist.Beta(alpha_prior, beta_prior))
        # pyro.sample("obs", dist.Binomial(n_samples, theta), obs=y)
        for i in range(len(y)): 
            # observe datapoint i using the likelihood
            pyro.sample("obs_{}".format(i), dist.Binomial(n_samples, theta), obs=y[i])
    def multi_model(y):
        mu = pyro.sample("theta", dist.Beta(alpha_prior, beta_prior))
        # pyro.sample("obs", dist.Binomial(n_samples, theta), obs=y)
        for i in range(len(y)):
            # observe datapoint i using the likelihood
            pyro.sample("obs_{}".format(i), dist.Binomial(n_samples, theta), obs=y[i])
    y_data = []

    single_guide = AutoNormal(single_model)
    multi_guide = AutoNormal(multi_model)
    # saves params in pyro as "AutoNormal.locs.mu" and "AutoNormal.scales.mu"
    optimizer = pyro.optim.Adam({}) # can leave args empty
    # {"lr": 0.02} is suggested by the vignettes at https://pyro.ai/examples/intro_long.html#Models-in-Pyro
    # lr = 0.005 and betas=(0.95, 0.999) are suggested at the vignette https://pyro.ai/examples/svi_part_i.html
    # default value of adam is lr=0.001, betas=(0.9, 0.999), eps=1e-08, weight_decay=0
    # ^^ taken frm torch.optim.Adam docs since pyro.optim.Adam is a thin wrapper around torch.optim.Adam

    single_elbo = Trace_ELBO()
    multi_elbo = Trace_ELBO(num_particles=100)
    single_svi = SVI(single_model, single_guide, optimizer, single_elbo)
    multi_svi = SVI(multi_model, multi_guide, optimizer, multi_elbo)

    def run_svi(y_data, svi_model, samps=1):
        tracker = {
            'mu_loc': np.zeros(max_iters),
            'mu_scale': np.zeros(max_iters)
        }
        # the 0th entry will be incorrect since 
        # the guide doesn't store the params in pyro 
        # until after the first step is called
        for i in tqdm(range(max_iters), position=0, leave=False, desc="{} MC samples".format(samps)):
            svi_model.step(y_data)
            tracker['mu_loc'][i] = pyro.param("AutoNormal.locs.theta").item()
            tracker['mu_scale'][i] = pyro.param("AutoNormal.scales.theta").item()
        return tracker
    single_tracker = run_svi(y_data, single_svi, samps=1)
    multi_tracker = run_svi(y_data, multi_svi, samps=100)
    return single_tracker, multi_tracker

In [ ]:
results = []
for seed in tqdm(range(20), position=1, desc="Random restarts..."):
    result = run_restart(seed)
    results.append(result)

Random restarts...:   0%|          | 0/20 [00:00<?, ?it/s]

1 MC samples:   0%|          | 0/100000 [00:00<?, ?it/s]

100 MC samples:   0%|          | 0/100000 [00:00<?, ?it/s]

In [ ]:
plt.rcParams.update({'font.size': 20})
# ----------------------------
# unpack results -> stacks also transform
# ----------------------------
single_means, single_stds = [], []
multi_means,  multi_stds  = [], []
for single_elbo, single_tracker, multi_elbo, multi_tracker in tqdm(results):
    # Use vectorized transformation for speed
    single_mu, single_sigma = _trace_locscale_to_theta_moments(
        single_tracker['mu_loc'], single_tracker['std_loc'], n_samples=10_000, seed=0
    )
    multi_mu, multi_sigma = _trace_locscale_to_theta_moments(
        multi_tracker['mu_loc'], multi_tracker['std_loc'], n_samples=10_000, seed=0
    )
    
    single_means.append(single_mu)
    single_stds.append(single_sigma)
    multi_means.append(multi_mu)
    multi_stds.append(multi_sigma)

single_means = np.stack(single_means, axis=0)  # (N, T)
single_stds  = np.stack(single_stds,  axis=0)  # (N, T)
multi_means  = np.stack(multi_means,  axis=0)  # (N, T)
multi_stds   = np.stack(multi_stds,   axis=0)  # (N, T)

N, T = single_means.shape
x = np.arange(T)

# If you have "best" from config, these should be on theta-scale (0,1)
best_mu = float(config["best_mean"]) if "config" in globals() and "best_mean" in config else None
best_std = float(config["best_std"]) if "config" in globals() and "best_std" in config else None
best_mu, best_std = _logistic_normal_mean_std_mc(best_mu, best_std, n_samples=50_000, seed=1)
# ----------------------------
# Plot 1: overlay a few trajectories from each condition
# ----------------------------
k = 2  # how many trajectories to show from each
idx = np.linspace(0, N - 1, k, dtype=int)  # deterministic selection

fig, axs = plt.subplots(2, 1, figsize=(12, 14))

# std of mu
for i in idx:
    axs[0].plot(x, single_stds[i], alpha=0.6, linewidth=1, color='blue')
for i in idx:
    axs[0].plot(x, multi_stds[i],  alpha=0.6, linewidth=1, color='green')

axs[0].axhline(best_std, color='red', linestyle='--', label=r'Best Variational approx of $\sigma_p$')
axs[0].set_title(r'Variational Approximation of $\sigma_p$ (Posterior Std of $\mu$): a few runs')
axs[0].set_xlabel('Iteration')
axs[0].set_ylabel(r'$\sigma_p$')
axs[0].grid()
axs[0].set_ylim(best_std * 0.6, best_std * 1.6)

# mean of mu
for i in idx:
    axs[1].plot(x, single_means[i], alpha=0.6, linewidth=1, color='blue', label=None)
for i in idx:
    axs[1].plot(x, multi_means[i],  alpha=0.6, linewidth=1, label=None)

axs[1].axhline(best_mu, color='red', linestyle='--', label=r'Best Variational approx of $\mu_p$')
axs[1].set_title(r'Variational Approximation of $\mu_p$ (Posterior Mean of $\mu$): a few runs')
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel(r'$\mu_p$')
axs[1].grid()
axs[1].set_ylim(best_mu - 3 * best_std, best_mu + 3 * best_std)

# manual legend (so we don’t get 16 duplicate entries)
axs[0].plot([], [], color='blue', label='1 MC sample (some runs)')
axs[0].plot([], [], color='green', label='100 MC samples (some runs)')
axs[0].legend()

axs[1].plot([], [], color='blue', label='1 MC sample (some runs)')
axs[1].plot([], [], color='green', label='100 MC samples (some runs)')
axs[1].legend()

plt.tight_layout()
plt.show()

# ----------------------------
# Plot 2: mean trajectory ± 1 std band across runs (separate plots for 1-MC and 100-MC)
# ----------------------------
def plot_mean_band(ax, x, Y, true_value, title, ylabel):
    """
    Y: (N, T) trajectories
    """
    m = Y.mean(axis=0)
    s = Y.std(axis=0)
    ax.plot(x, m, linewidth=2, label='Mean across runs')
    ax.fill_between(x, m - s, m + s, alpha=0.25)
    ax.axhline(true_value, color='red', linestyle='--', label='Best value')
    ax.set_title(title)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(ylabel)
    ax.grid()
    ax.legend()

# 1 MC sample: mean ± sd across runs
fig, axs = plt.subplots(2, 1, figsize=(12, 14))
plot_mean_band(
    axs[0], x, single_stds, best_std,
    r'1 MC sample: $\sigma_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\sigma_p$'
)
axs[0].set_ylim(best_std * 0.6, best_std * 1.6)

plot_mean_band(
    axs[1], x, single_means, best_mu,
    r'1 MC sample: $\mu_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\mu_p$'
)
axs[1].set_ylim(best_mu - 3 * best_std, best_mu + 3 * best_std)

plt.tight_layout()
plt.show()

# 100 MC samples: mean ± sd across runs
fig, axs = plt.subplots(2, 1, figsize=(12, 14))
plot_mean_band(
    axs[0], x, multi_stds, best_std,
    r'100 MC samples: $\sigma_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\sigma_p$'
)
axs[0].set_ylim(best_std * 0.6, best_std * 1.6)

plot_mean_band(
    axs[1], x, multi_means, best_mu,
    r'100 MC samples: $\mu_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\mu_p$'
)
axs[1].set_ylim(best_mu - 3 * best_std, best_mu + 3 * best_std)

plt.tight_layout()
plt.show()


In [16]:
for i in pyro.get_param_store():
    print(i)

AutoNormal.locs.theta
AutoNormal.scales.theta
